# $K_S$ veto
## Calculate systematic uncertainty due to $K_S$ veto

### Function that converts $K_i$ to $R_i$

In [1]:
std::vector<double> ConvertKiToRi(const std::vector<double> &Ki,
                                  const std::vector<double> &Kbari) {
    std::vector<double> MergedKi;
    const std::size_t N = Ki.size();
    for(std::size_t i = 0; i < N; i++) {
        MergedKi.push_back(Kbari[N - i - 1]);
    }
    for(std::size_t i = 0; i < N; i++) {
        MergedKi.push_back(Ki[i]);
    }
    std::vector<double> Ri;
    for(std::size_t i = 0; i < 2*N - 1; i++) {
        Ri.push_back(MergedKi[i]);
        if(i != 0) {
            double Sum = 0;
            for(std::size_t j = i; j < 2*N; j++) {
                Sum += MergedKi[j];
            }
            Ri[i] /= Sum;
        }
    }
    return Ri;
}

### Function that parses the $K_i$, $c_i$ and $s_i$ into a single ```std::vector<double>```

In [2]:
std::vector<double> ParseDParameters(const std::string &Filename) {
    std::ifstream File(Filename);
    std::string Line;
    std::getline(File, Line);
    std::vector<double> Ki, Kbari, ci, si;
    for(std::size_t i = 0; i < 4; i++) {
        std::getline(File, Line);
        std::replace(Line.begin(), Line.end(), ',', ' ');
        double K, Kbar, c, s, dummy;
        std::stringstream ss(Line);
        ss >> dummy >> K >> Kbar >> c >> s;
        Ki.push_back(K);
        Kbari.push_back(Kbar);
        ci.push_back(c);
        si.push_back(s);
    }
    std::vector<double> Ri = ConvertKiToRi(Ki, Kbari);
    std::vector<double> DParameters(ci);
    DParameters.insert(DParameters.end(), si.begin(), si.end());
    DParameters.insert(DParameters.end(), Ri.begin(), Ri.end());
    // KKpipi BF
    DParameters.insert(DParameters.begin(), 0.0);
    // KKpipi BF with KLpipi
    DParameters.insert(DParameters.end(), 0.0);
    // rDcosDeltaKpi and rDsinDeltaKpi
    DParameters.insert(DParameters.end(), 0.0);
    DParameters.insert(DParameters.end(), 0.0);
    return DParameters;
}

### Load model predicted values

In [3]:
const std::string Directory("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins/ModelPredictions/");
const auto WithoutVeto = ParseDParameters(Directory + "cisi_LHCbModel_NoKSVeto.csv");
const auto WithVeto = ParseDParameters(Directory + "cisi_LHCbModel_WithKSVeto.csv");

### Find the difference

In [4]:
std::vector<double> Diff;
for(std::size_t i = 0; i < WithVeto.size(); i++) {
    Diff.push_back(WithoutVeto[i] - WithVeto[i]);
}

### Construct covariance matrix

In [7]:
TFile File("KSVeto_Systematics_CovarianceMatrix.root", "RECREATE");
TMatrixT<double> CovMatrix(Diff.size(), Diff.size());
for(std::size_t i = 0; i < Diff.size(); i++) {
    for(std::size_t j = 0; j < Diff.size(); j++) {
        CovMatrix(i, j) = Diff[i]*Diff[j];
    }
}
File.WriteObject(&CovMatrix, "CovMatrix");
File.Close();